In [ ]:
import sys
from pathlib import Path
module_path = str(Path('..').resolve())
if module_path not in sys.path:
    print(f"added {module_path} to path")
    sys.path.append(module_path)
for line in sys.path:
    print(line)

import pyfar as pf
import numpy as np
from numpy.typing import NDArray
import matplotlib.pyplot as plt
from IPython.display import Audio, display
from algorithms.common import chunked
from algorithms.avendano import ambiance_index, panning_index
from algorithms.gomez import panning_histogram
import librosa
from scipy.signal import ShortTimeFFT, get_window
import matplot2tikz
from python_auditory_toolbox import auditory_toolbox as pat
import scipy
plt.rcParams['font.family'] = "Times"
# plt.rcParams['text.usetex'] = True

%load_ext jupyter_tikz
%load_ext autoreload
%autoreload 2

In [ ]:
def to_dat(X, Y):
    return '\n'.join([f"{x} {y}" for x,y in zip(X,Y)])

In [ ]:
hrirs, sources = pf.signals.files.head_related_impulse_responses(position='horizontal', diffuse_field_compensation=True)

In [ ]:
elevation = 0
azimuth_L = np.pi/3 # 60 degrees left
azimuth_R = -np.pi/3 # 60 degrees right
radius = 1

desired_direction_L = pf.Coordinates.from_spherical_elevation( azimuth_L, elevation, radius)
desired_direction_R = pf.Coordinates.from_spherical_elevation( azimuth_R, elevation, radius)

index_L, _ = sources.find_nearest(desired_direction_L)
index_R, _ = sources.find_nearest(desired_direction_R)

sources.show([index_L, index_R])
plt.show()


In [ ]:
ax = pf.plot.freq_group_delay(hrirs[index_R], label=["Left ear", "Right ear"])
# ax[0].legend()
# ax[1].legend()
plt.show()

In [ ]:
kick = pf.io.read_audio(Path.home() / "Music/freesound/581451__johnnydekk__fractanimal_acoustic_drum_kit_kick_2.wav")
hihat = pf.io.read_audio(Path.home() / "Music/freesound/581473__johnnydekk__fractanimal_acoustic_drum_kit_high_hat_3.wav")
# ax_kick = pf.plot.time_freq(kick)
ax_hihat = pf.plot.time_freq(hihat)
plt.show()

In [ ]:
fig, ax = plt.subplots(2,1, figsize=(8,4), sharex=True, layout='tight')



def render(audio, hrir, ax, slug):
    binaural_audio = pf.dsp.convolve(audio, hrir)
    cropped = pf.dsp.time_crop(binaural_audio, [0,0.03], unit='s')
    cropped.time[0] += 1
    cropped.time[1] -= 1
    cropped.sampling_rate /= 1000
    # cropped.times *= 1000
    X = (cropped.times * 1000)

    print("ITD", X[np.argmax(cropped.time, axis=1)])
    print("MAX", np.max(cropped.time, axis=1))
    print("MIN", np.min(cropped.time, axis=1))

    Path(f"../../papers/kappa/figures/{slug}_L.dat").write_text(to_dat(X, cropped.time[0]))
    Path(f"../../papers/kappa/figures/{slug}_R.dat").write_text(to_dat(X, cropped.time[1]))

    pf.plot.time(cropped, label=["Left ear", "Right ear"], ax=ax, color='k')
    ax.set_ylim(-2,2)

# def latex_plot_hihat():
    # txt = '\\begin{tikzpicture}'

    # txt += '\\end{tikzpicture}'

    

render(hihat, hrirs[index_R], ax[0], "hihat")
render(kick, hrirs[index_R], ax[1], "kick")

ax[0].set_xlabel(None)
ax[1].set_xlabel("Time (ms)")
ax[0].set_ylabel("Hi-hat")
ax[1].set_ylabel("Kick drum")
plt.show()

tikz.save(
    "../../papers/kappa/tests/matplot2tikz.tex", 
    figure=fig, 
    axis_width='\\textwidth', 
    axis_height='4cm',
)

In [ ]:
music = pf.io.read_audio("/Users/richard/Music/archive/09. Tsubasao Kudasai.flac", stop=int(10. * 44100))
pf.plot.spectrogram(music, freq_scale='log')
plt.show()

In [ ]:
from matplotlib import colormaps, color_sequences

# def amplitude_pan(sig: pf.Signal, pan: float) -> pf.Signal:
#     assert -1 <= pan <= 1.
#     assert sig.cshape == (2,)
#     a = (pan + 1) / 2
#     al, ar = np.cos(a * 0.5 * np.pi), np.sin(a * 0.5 * np.pi)
#     print(a, al, ar)
#     left, right = sig.time[0] * al, sig.time[1] * ar
#     return pf.Signal(np.stack([left, right]), sig.sampling_rate)

# def downmix(sig: pf.Signal) -> pf.Signal:
#     assert sig.cshape == (2,)
#     a = 1/np.sqrt(2)
#     data = np.sum(sig.time, axis=0) * a
#     return pf.Signal(np.stack([data, data]), sig.sampling_rate)

# def downmix_to_mono(sig: pf.Signal) -> pf.Signal:
#     assert sig.cshape == (2,)
#     a = 1/np.sqrt(2)
#     data = np.sum(sig.time, axis=0) * a
#     return pf.Signal(data, sig.sampling_rate)


# def make_panned_sound(pan: float = 0.5):
#     fvocals = "/Users/richard/Music/freesound/245325__vocalpercussion__vocal_percussion_deh_deh_boom.wav"
#     fguitar = "/Users/richard/Music/freesound/474398__bangcorrupt__blues_guitar_intro.flac"
#     ftrumpet = "/Users/richard/Music/freesound/2323__synapse__trumpetloop01.aiff"

#     def _load(f):
#         audio = pf.io.read_audio(f)
#         if audio.sampling_rate != 48000:
#             audio = pf.dsp.resample(audio, sampling_rate=48000)
#         audio = pf.Signal(audio.time[0, :480000], sampling_rate=audio.sampling_rate)
#         audio = pf.dsp.normalize(audio, reference_method='energy', target=0.5)
#         return audio
    
#     vocals = _load(fvocals)
#     guitar = _load(fguitar)
#     trumpet = _load(ftrumpet)
#     # sound1 = amplitude_pan(downmix(pf.dsp.time_crop(pf.dsp.resample(pf.io.read_audio(f1), sampling_rate=48000), [0.,10.], unit='s')), -pan)
#     # sound2 = amplitude_pan(downmix(pf.dsp.time_crop(pf.dsp.resample(pf.io.read_audio(f2), sampling_rate=48000), [0.,10.], unit='s')), pan)
#     # sound = sound1 + sound2
#     print(vocals, guitar, trumpet)
#     # left = vocals * 0.5 + guitar * 0.7 + trumpet * 0.1
#     # right = vocals * 0.5 + guitar * 0.3 + trumpet * 0.9
#     left = guitar * 0.7 + trumpet * 0.1
#     right = guitar * 0.3 + trumpet * 0.9
#     sound = pf.Signal(np.stack([left.time[0], right.time[0]]), 48000)
#     return sound

def get_music():
    # f = "/Users/richard/Music/archive/09. Tsubasao Kudasai.flac"
    # return pf.dsp.time_crop(pf.dsp.resample(pf.io.read_audio(f), sampling_rate=48000), [0,480000])
    f = "/Users/richard/Music/projects/panning_and_reverb/panning_and_reverb_wav.wav"
    return pf.io.read_audio(f)

sound = get_music()

# freqs, times, spect = pf.dsp.spectrogram(sound)
fft_size = 1024
hop_size = 512
stft = ShortTimeFFT(
    get_window("hann", fft_size),
    hop_size,
    sound.sampling_rate
)

spect = stft.stft(sound.time)
spect = np.swapaxes(spect, 1, 2)
times = np.arange(spect.shape[1]) * (stft.hop / stft.fs)
freqs = stft.f

mosaic = """
aaaaaaaaaaaaaaaaaaaaaaaaaa.
aaaaaaaaaaaaaaaaaaaaaaaaaa.
bbbbbbbbbbbbbbbbbbbbbbbbbbB
bbbbbbbbbbbbbbbbbbbbbbbbbbB
bbbbbbbbbbbbbbbbbbbbbbbbbbB
ccccccccccccccccccccccccccC
ccccccccccccccccccccccccccC
ccccccccccccccccccccccccccC
"""
fig, ax = plt.subplot_mosaic(mosaic, figsize=(8,5), layout='tight')
# fig = plt.figure()
# ax = dict()
# ax["a"] = fig.add_subplot(3, 1, 1)
# ax["b"] = fig.add_subplot(3, 1, 2, sharex=ax["a"])
# ax["c"] = fig.add_subplot(3, 1, 3, sharex=ax["a"])

ai = ambiance_index(spect, lamb=0.85)
pi = panning_index(spect)


def decibels(x, mindb=-70.):
    epsilon = np.nextafter(np.float64(0), np.float64(1))
    # epsilon = 1e-16
    return np.clip(20 * np.log10(x + epsilon), a_min=mindb, a_max=float("inf"))

def make_peaks(sound, pps):
    n_sections = int((sound.n_samples * pps) / sound.sampling_rate)
    channel_peaks = []
    for channel in sound.time:
        chunks = np.array_split(channel, n_sections)
        chunks[-2] = np.append(chunks[-2], chunks[-1])
        del chunks[-1]
        channel_peaks.append([np.max(np.abs(chunk)) for chunk in chunks])
    return pf.Signal(np.stack(channel_peaks), pps)

def plot_stereo(sound: pf.Signal, pps: int, ax: plt.Axes, colormap: str = 'berlin'):
    peaks = make_peaks(sound, pps)
    offset = np.max(peaks.time)
    x = np.arange(peaks.n_samples) / peaks.sampling_rate
    color_l, color_r = colormaps.get(colormap)([0.1,0.9])
    ax.fill_between(x, peaks.time[0] + offset, -peaks.time[0] + offset, color=color_l)
    im = ax.fill_between(x, peaks.time[1] - offset, -peaks.time[1] - offset, color=color_r)
    ax.set_yticks([offset, -offset], labels=["left", "right"])
    ax.set_xlim(x[[0,-1]])
    ax.set_ylim([-0.8,0.8])
    ax.set_xticks([])
    return im


weights = np.sum(np.abs(spect), axis=0)
panogram = np.vstack([np.histogram(pi[i], bins=100, range=[-1,1], weights=weights[i])[0] for i in range(len(pi))])

plot_stereo(sound, 400, ax['a'])
# ax['a'].set_title("Audio waveform")
# ax['a'].set_xlabel("Time (s)")

extent = (times[0], times[-1], freqs[0], freqs[-1])
display_ai = np.nan_to_num(decibels(np.rot90(ai)))
display_pi = np.rot90(pi)
display_panogram = decibels(np.rot90(panogram))

# print(display_ai.min(), display_ai.max())
# print(display_pi.min(), display_pi.max())


img_ai = ax['b'].imshow(display_ai, aspect='auto', cmap='grey', extent=extent)
ax['b'].set_ylabel("Frequency (Hz)")
ax['b'].set_yticks([0,5000,10000,15000,20000], labels=["0", "5k", "10k", "15k", "20k"])
# ax['b'].set_xlabel("Time (s)")
# ax['b'].set_title("Ambience Index", y=1.0, x=0, pad=-14)

ax['b'].set_xticks([])


img_pi = ax['c'].imshow(display_pi, aspect='auto', cmap='berlin', extent=extent)
ax['c'].set_ylabel("Frequency (Hz)")
ax['c'].set_yticks([0,5000,10000,15000,20000], labels=["0", "5k", "10k", "15k", "20k"])
ax['c'].set_xlabel("Time (s)")
# ax['c'].set_title("Panning Index", y=1.0, x=0, pad=-14)

cbar = fig.colorbar(img_ai, cax=ax["B"])
cbar.ax.set_yticks([display_ai.min(), display_ai.max()], labels=[0,1])


cbar = fig.colorbar(img_pi, cax=ax["C"], ticks=[display_pi.min(), display_pi.max()])
cbar.ax.set_yticks([display_pi.min(), 0, display_pi.max()], labels=["Left", "Centre", "Right"])



props = dict(boxstyle='round', facecolor='white', alpha=0.8)
props2 = dict(boxstyle='round', facecolor='white', alpha=0.9)
x = 0.01
y = 0.94
ax["a"].text(x, y, "Audio Waveform", transform=ax["a"].transAxes, verticalalignment='top', bbox=props)
ax["b"].text(x, y, "Ambience Index", transform=ax["b"].transAxes, verticalalignment='top', bbox=props2)
ax["c"].text(x, y, "Panning Index", transform=ax["c"].transAxes, verticalalignment='top', bbox=props2)

fig.savefig("../../papers/kappa/figures/waveforms.pdf")

Audio(sound.time, rate=sound.sampling_rate)

In [ ]:
new_length_400pps = int((sound.n_samples * 400) / sound.sampling_rate)
new_length_10pps = int((sound.n_samples * 10) / sound.sampling_rate)
new_length_1pps = int(sound.n_samples / sound.sampling_rate)

print(sound.n_samples, new_length_400pps, new_length_10pps, new_length_1pps)

def make_peaks(sound, pps):
    n_sections = int((sound.n_samples * pps) / sound.sampling_rate)
    channel_peaks = []
    for channel in sound.time:
        chunks = np.array_split(channel, n_sections)
        chunks[-2] = np.append(chunks[-2], chunks[-1])
        del chunks[-1]
        channel_peaks.append([np.max(np.abs(chunk)) for chunk in chunks])
    return pf.Signal(np.stack(channel_peaks), pps)

def plot_stereo(sound: pf.Signal, pps: int, ax: plt.Axes, colormap: str = 'berlin'):
    peaks = make_peaks(sound, pps)
    offset = np.max(peaks.time)
    x = np.arange(peaks.n_samples) / peaks.sampling_rate
    color_l, color_r = colormaps.get(colormap)([0.1,0.9])
    ax.fill_between(x, peaks.time[0] + offset, -peaks.time[0] + offset, color=color_l)
    ax.fill_between(x, peaks.time[1] - offset, -peaks.time[1] - offset, color=color_r)
    ax.set_yticks([])


fig, ax = plt.subplots(3,1, figsize=(10, 10))

plot_stereo(sound, 400, ax[0])
plot_stereo(sound, 10, ax[1])
plot_stereo(sound, 1, ax[2])



In [ ]:
nx, ny = (3,4)
idx = np.mgrid[:nx,:ny].reshape(2, -1).T
frame = [64,256,1024,4096]
hop = [0.5, 0.75, 0.875]


s = 3
fig, ax = plt.subplots(ny, nx, figsize=(nx*s, ny*s), sharex=True)

# file = "659528__johaynes__jos-genedrums-150bpm-42.wav"
# file = "581473__johnnydekk__fractanimal_acoustic_drum_kit_high_hat_3.wav"
file = "245325__vocalpercussion__vocal_percussion_deh_deh_boom.wav"
transient_signal = downmix_to_mono(pf.io.read_audio(f"/Users/richard/Music/freesound/{file}"))
transient_signal = pf.dsp.time_crop(transient_signal, [0.85,1.1], unit='s')

print(transient_signal)
for i,j in idx:
    f = frame[j]
    h = hop[i]
    a = ax[j,i]
    pf.plot.spectrogram(transient_signal, window_length=f, window_overlap_fct=h, ax=a, colorbar=False)
    a.set_title(f"{f}, {int(f*(1-h))}")
    a.set_xlabel(None)
    a.set_ylabel(None)

fig.savefig("spectrogram_accuracy.pdf")

In [ ]:
import librosa
audio, sr = librosa.load(librosa.util.example('trumpet'))
fig, ax = plt.subplots(2,1)

D = librosa.amplitude_to_db(np.abs(librosa.stft(audio)), ref=np.max)
S = librosa.feature.melspectrogram(S=D, sr=sr, n_mels=32)

librosa.display.specshow(data=D, sr=sr, ax=ax[0], y_axis='linear')
librosa.display.specshow(S, sr=sr, ax=ax[1], y_axis='mel')

fig.savefig("mel.pdf")


In [ ]:


fig, ax = plt.subplots(3,1, figsize=(8, 8), sharex=True)

# audio, sr = librosa.load(librosa.util.example('trumpet', hq=True))
# signal = pf.signals.exponential_sweep_time(48000, [20,20000], sampling_rate=48000)
signal = pf.io.read_audio("/Users/richard/Music/freesound/31912__acclivity__globalwarming_marg.wav")
audio, sr = signal.time[0], signal.sampling_rate

start, end = 0.8, 2.6
duration = end - start
audio = audio[int(start * sr):int(end * sr)]

stft_bands = 2048
num_bands = 32

def kilo_formatter(x):
    if x < 1000:
        return str(int(x))
    else:
        return f"{int(x/1000)}k"


linear_freqs = np.fft.rfftfreq(stft_bands, d=1/sr)
erb_freqs = pat.ErbSpace(low_freq=100, high_freq=sr/2, n=num_bands)[::-1]

coefs = pat.MakeErbFilters(sr, num_bands, low_freq=100)
y = pat.ErbFilterBank(audio, coefs)
y *= np.sign(y) # half wave rectify
y = np.sqrt(y) # square root

# low pass
fc = 800
b, a = scipy.signal.butter(4, fc/sr, btype='low', analog=False)
y = scipy.signal.filtfilt(b, a, y)

coch_chunk_length = 0.02
chunk_size = int(coch_chunk_length * sr)
hop_size = int(0.5 * chunk_size)
chunks = np.lib.stride_tricks.sliding_window_view(y, chunk_size, axis=1)[:, ::hop_size, :]
print(chunks.shape)
# coch = np.sqrt(np.sum(np.pow(chunks, 2), axis=(2,))) # RMS
coch = np.mean(chunks, axis=2)


D = librosa.amplitude_to_db(np.abs(librosa.stft(audio, n_fft=stft_bands)), ref=np.max)
S = librosa.feature.melspectrogram(S=D, sr=sr, n_mels=num_bands)
mel_freqs = librosa.mel_frequencies(n_mels=num_bands, fmax=sr/2)
print(D.shape, S.shape, coch.shape)


images = []

xmax = D.shape[1] - 1
images.append(ax[0].imshow(np.flipud(D), aspect='auto', cmap='magma', extent=(0, duration, 0, linear_freqs[-1])))
images.append(ax[1].imshow(np.flipud(S), aspect='auto', cmap='magma', extent=(0, duration, 0, num_bands)))
images.append(ax[2].imshow(librosa.amplitude_to_db(coch), aspect='auto', cmap='magma', extent=(0, duration, 0, num_bands)))


ax[0].set_ylabel("Frequency (Hz)")
ax[0].set_yticks([0,5000,10000,15000,20000], labels=["0", "5k", "10k", "15k", "20k"])

desired_ticks = [100, 500, 1000, 2000, 5000, 10000, 20000]

ax[1].set_ylabel("Frequency (Hz)")
yticks = np.interp(desired_ticks, mel_freqs, np.arange(num_bands))
print(yticks)
ax[1].set_yticks(yticks, labels=[kilo_formatter(x) for x in desired_ticks])

ax[2].set_ylabel("Frequency (Hz)")
yticks = np.interp(desired_ticks, erb_freqs, np.arange(num_bands))
ax[2].set_yticks(yticks, labels=[kilo_formatter(x) for x in desired_ticks])
ax[2].set_xlabel("Time (s)")

props = dict(boxstyle='round', facecolor='white', alpha=0.9)
x = 0.02
y = 0.94
ax[0].text(x, y, "Spectrogram", transform=ax[0].transAxes, verticalalignment='top', bbox=props)
ax[1].text(x, y, "Mel Spectrogram", transform=ax[1].transAxes, verticalalignment='top', bbox=props)
ax[2].text(x, y, "Cochleagram", transform=ax[2].transAxes, verticalalignment='top', bbox=props)


fig.colorbar(images[0], ax=ax, label="Sound intensity (dB)")
fig.savefig("../../papers/kappa/figures/spect_comp.pdf")

plt.show()

In [ ]:
fs = 44100
fft_length = 4096

fmin = 20
num_bands = 8

freqs = np.fft.rfftfreq(fft_length, d=1/fs)
F = np.tile(freqs, (num_bands,1))
print(F.shape)

mel_filters = librosa.filters.mel(sr=fs, n_fft=fft_length, n_mels=num_bands, fmin=fmin)
mel_filters /= np.max(mel_filters)
# mel_filters[mel_filters == 0] = np.nan # showing all the zero weights looks cluttered, so just get rid of them


fig, ax = plt.subplots(1,2, figsize=(8,3))
ax[0].plot(F.T, mel_filters.T, color='k')
ax[1].semilogx(F.T, mel_filters.T, color='k')


tikz_fig = r"""
\begin{tikzpicture}

\definecolor{darkgray176}{RGB}{176,176,176}

\begin{groupplot}[
    group style={
        group size=2 by 1,
        horizontal sep=10pt,
    },
    xlabel={Frequency (Hz)},
]

\nextgroupplot[
    xmin=-2000,
    xmax=25000,
    xtick={0, 5000, 10000, 15000, 20000},
    xticklabels={{0}, {5k}, {10k}, {15k}, {20k}},
    scaled x ticks=false,
    ylabel={Filter weights},
    title={Mel filterbank (linear frequency)},
]
PLOT1
LABELS1

\nextgroupplot[
    xmode=log,
    log basis x={10},
    xmin=15,
    xmax=50000,
    xtick={20, 200, 2000, 20000},
    xticklabels={{20}, {200}, {2k}, {20k}},
    yticklabels=\empty,
    title={Mel filterbank (logarithmic frequency)},
]
PLOT2
LABELS2

\end{groupplot}

\end{tikzpicture}
"""

def to_table(X, Y):
    table = ["table {%"]
    for x,y in zip(X, Y):
        if y>0:
            table.append(f"{x:.3f} {y:.3f}")
    table.append("}")
    return table

def to_plot(table):
    table_txt = '\n'.join(table)
    return f"\\addplot [black]\n{table_txt};"


# tikz_fig = matplot2tikz.get_tikz_code(fig)
# print(tikz_fig)

plot1 = []
plot2 = []
labels1 = []
labels2 = []

for n, filt in enumerate(mel_filters):
    x = F[0][np.argmax(filt)]
    y = np.max(filt)
    print(n, x, y)
    lin_anchor = "south west" if n in [1,2] else "south"
    log_anchor = "south"
    lin_label = f"\\node[draw=black,circle,fill=white] at (axis cs:{x:.3f},{y:.3f}) [anchor={lin_anchor}, inner sep=2pt, outer sep=4pt] {{{n}}};"
    log_label = f"\\node[draw=black,circle,fill=white] at (axis cs:{x:.3f},{y:.3f}) [anchor={log_anchor}, inner sep=2pt, outer sep=4pt] {{{n}}};"
    # labels1.append(lin_label)
    # labels2.append(log_label)

    plot = to_plot(to_table(F[0], filt))
    plot1.append(plot)
    plot2.append(plot)

    # ax[0].text(x, y, f"n={n}", transform=ax[0].transData)
    # ax[1].text(x, y, f"n={n}", transform=ax[1].transAxes)
    # ax[0].text(1, 0, f"fuck", transform=ax[0].transData)

# labels = "\n".join(labels)
tikz_fig = tikz_fig.replace("PLOT1", "\n".join(plot1))
tikz_fig = tikz_fig.replace("PLOT2", "\n".join(plot2))
tikz_fig = tikz_fig.replace("LABELS1", "\n".join(labels1))
tikz_fig = tikz_fig.replace("LABELS2", "\n".join(labels2))
# tikz_fig.insert(tikz_fig.find(), labels)

# print(plot1[0])

preamble = r"""
\usepackage{tikz}
\usepackage{pgfplots}
\usetikzlibrary{calc,arrows.meta}
\usepgfplotslibrary{groupplots}
\pgfplotsset{compat=1.18}
"""

Path("../../papers/kappa/figures/mel_filterbank.tikz").write_text(tikz_fig)

# %tikz -e -t=pgfplots -l="calc,arrows.meta" -lp="groupplots" -p="$preamble" "$tikz_fig"
%tikz -e -p "$preamble" "$tikz_fig"

# fig.savefig("test.pdf")

# plt.show()


# mel_filters = librosa.amplitude_to_db(mel_filters) # display in decibles to make it similar to the gammatone filters
# print(librosa.filters.mel_frequencies(n_mels=8, fmax=24000, fmin=fmin))

# fig, ax = plt.subplots(2,2, figsize=(8, 6), sharey=True)


# ax[0,0].plot(F.T, mel_filters.T, color='k')
# ax[0,0].set_xscale('log')
# ax[0,0].set_ylabel("Amplitude")
# ax[0,0].set_xlabel("Frequency (Hz)")
# ax[0,0].set_title("Mel filters (logarithmic)")
# ax[0,1].plot(F.T, mel_filters.T, color='k')
# ax[0,1].set_xscale('linear')
# # ax[1].set_ylabel("Amplitude")
# ax[0,1].set_xlabel("Frequency (Hz)")
# ax[0,1].set_title("Mel filters (linear)")

# ax[0,0].set_ylim([-60, 0])
# ax[0,0].set_xlim(20, fs/2)
# ax[0,1].set_ylim([-60, 0])
# ax[0,1].set_xlim(20, fs/2)

# fig.tight_layout()
# fig.savefig("mel_filters.pdf")

# print(mel_filters.shape, freqs.shape)

# # for i in range(len(mel_filters)):
#     # Path(f"../../papers/kappa/figures/mel_{i}.dat").write_text(to_dat(freqs, mel_filters[i]))



# # Now plot gammatone filters
# fcoefs = pat.MakeErbFilters(fs, 8, low_freq=fmin)
# impulse_len = fft_length
# x = np.zeros(impulse_len)
# x[0] = 1
# y = pat.ErbFilterBank(x, fcoefs)
# take_len = impulse_len // 2
# resp = 20*np.log10(np.abs(np.fft.fft(y.T, axis=0)))[:take_len]
# ax[1,0].semilogx(np.arange(take_len)*(fs/impulse_len),resp, color='k')
# ax[1,0].set_xlim(20, fs/2)
# ax[1,0].set_ylim(-60, 0)
# ax[1,0].set_ylabel("Filter response (dB)")
# ax[1,0].set_xlabel("Frequency (Hz)")


# gammatone_freq = np.arange(take_len)*(fs/impulse_len)
# for i in range(resp.shape[1]):
#     mask = resp[:, i] >= -60
#     Path(f"../../papers/kappa/figures/gammatone_{i}.dat").write_text(to_dat(gammatone_freq[mask], resp[:,i][mask]))



In [ ]:
import scipy
a = 1.99
rv = scipy.stats.gamma(a)
x = np.linspace(0, 10, 200)
y = rv.pdf(x) * np.cos(8*x) - 0.5
y2 = rv.pdf(x-1) * np.cos(8*(x-1)) + 0.5
y3 = (y*0.5) + 0.75
plt.plot(x, y)
plt.plot(x, y2)
plt.plot(x, y3)

def write_path(x, y):
    coords = " ".join([f"({a[0]:.3f},{a[1]:.3f})" for a in  zip(x,y)])
    return f"\\addplot [domain=0:10, no markers, smooth] coordinates {{{coords}}};"

Path("itd_R.tex").write_text(write_path(x,y))
Path("itd_L.tex").write_text(write_path(x,y2))
Path("ild_R.tex").write_text(write_path(x,y))
Path("ild_L.tex").write_text(write_path(x,y3))

print(max(rv.pdf(x)))


In [ ]:
def plot_hrir():
    elevation = 0
    azimuth_L = np.pi/6
    azimuth_R = -azimuth_L
    radius = 1

    desired_direction_L = pf.Coordinates.from_spherical_elevation( azimuth_L, elevation, radius)
    desired_direction_R = pf.Coordinates.from_spherical_elevation( azimuth_R, elevation, radius)
    index_L, _ = sources.find_nearest(desired_direction_L)
    index_R, _ = sources.find_nearest(desired_direction_R)
    print(desired_direction_L.azimuth, desired_direction_R.azimuth)


    dur = 64
    start = 10
    sig = pf.dsp.time_crop(hrirs[index_R], [start,start+dur])
    pf.plot.time(sig)

    X = np.arange(dur)
    Y = sig.time

    def txt(Y, name):
        points = "--".join([f"({x:.3f},{y:.3f})" for x,y in zip(X,Y)])
        return f"\\path[local bounding box={name}, draw=black] {points};"
    
    # print(txt(Y[0]))
    # print(txt(Y[1]))

    Path("../../papers/kappa/figures/brir_L.tex").write_text(txt(Y[0], "left_brir"))
    Path("../../papers/kappa/figures/brir_R.tex").write_text(txt(Y[1], "right_brir"))

plot_hrir()

In [ ]:
# def spiral(theta, a, b):
#     r = a * np.exp(b * theta)
#     x = np.cos(r)
#     y = np.sin(r)
#     return x, y

def cart2pol(xy):
    rho = np.sqrt(xy.real**2 + xy.imag**2)
    phi = np.arctan2(xy.imag, xy.real)
    return np.array([rho, phi])

def pol2cart(rp):
    rho, phi = rp
    x = rho * np.cos(phi)
    y = rho * np.sin(phi)
    return x + y * 1j

def spiral(a, b, t):
    print(t.shape)
    spir = a * np.exp((b + 1j) * t)
    spir.real *= -1
    return spir
    # rho, phi = cart2pol(spir)
    # # phi += np.pi/2
    # return pol2cart([rho, phi])

def coord(xy):
    return f"({xy.real:.3f},{xy.imag:.3f})"

def plot_cochlea(t, a=1, b=0.08, tension=0.618):
    d = 0.18
    inner = spiral(a, b, t)
    outer1 = spiral(a * (1+d), b, t)
    outer2 = spiral(a * (1-d), b, t)

    freqs = [200, 400, 800, 1600, 3200, 6400, 12000, 20000]
    log_pos = np.log10(freqs)
    log_pos -= min(log_pos)
    log_pos /= max(log_pos)
    log_pos_t = np.interp(log_pos, np.linspace(-0.01, 1.01, len(t)), t)

    base_label = spiral(a, b, t[-1:] + 0.168)[0]
    apex_label = inner[0] + 0.618 - 0.2j

    basilar_membrane_label1 = spiral(a, b, t[-4:-3])[0]
    basilar_membrane_label2 = pol2cart(cart2pol(basilar_membrane_label1) + [1.618, 0])

    frq_marks = spiral(a, b, log_pos_t)

    def path(XY, colour):
        points = " ".join([f"({x:.3f},{y:.3f})" for x,y in zip(XY.real, XY.imag)])
        return f"\\draw[{colour}] plot [smooth, tension={tension}] coordinates {{ {points} }};"

    def txt():
        txt = "\\begin{tikzpicture}"
        txt += path(inner, "black, dashed")
        txt += path(outer1, "black, thick")
        txt += path(outer2, "black, thick")
        txt += f"\\draw[black, thick] {coord(outer1[0])} to[bend right=90, looseness=1.618] {coord(outer2[0])};"
        txt += f"\\draw[black, thick] {coord(outer1[-1])} to[bend left=90, looseness=1] {coord(inner[-1])};"
        txt += f"\\draw[black, thick] {coord(outer2[-1])} to[bend right=90, looseness=1] {coord(inner[-1])};"

        txt += f"\\node[] (base_label) at {coord(base_label)} {{Base}};"
        txt += f"\\node[] (apex_label) at {coord(apex_label)} {{Apex}};"
        txt += f"\\draw[latex-] {coord(basilar_membrane_label1)} -- {coord(basilar_membrane_label2)};"
        txt += f"\\node[anchor=north] (basilar_membrane_label) at {coord(basilar_membrane_label2)} {{Basilar membrane}};"
        for pos,frq in zip(frq_marks, freqs):
            txt += f"\\node[circle, fill=black, draw=black, minimum size=4pt, text width=0pt, inner sep=0] (point_{frq}) at {coord(pos)} {{}};"
            txt += f"\\node[anchor=south, fill=white, draw=black, outer sep=4pt] (point_{frq}) at {coord(pos)} {{{frq}}};"

        txt += "\\end{tikzpicture}"
        return txt.replace('\\', '\n\\')

    return txt()

drawing = plot_cochlea(np.linspace(0, 18, 50))
# print(drawing)
Path("../../papers/kappa/figures/cochlea.tex").write_text(drawing)


%tikz -e "$drawing"
    


In [ ]:
# f = "/Users/richard/Music/projects/panning_and_reverb/panning_and_reverb_wav.wav"
f = "/Users/richard/Music/archive/09. Tsubasao Kudasai.flac"
music = pf.dsp.time_crop(pf.io.read_audio(f), [0, 30], unit='s')

chunks = chunked(music, 1024, 512)
spect = np.fft.rfft(chunks)
print(spect.shape)

def __stats(x):
    print(f"{np.min(x)=} {np.max(x)=} {np.mean(x)=} {np.std(x)=}")


hist = panning_histogram(spect, 100)

plt.plot(hist)
plt.show()

In [ ]:
number_of_turns = 3
max_angle = number_of_turns * 2 * np.pi
t = np.linspace(0, 1, 100)
starting_angle = -np.pi/2
theta = starting_angle + (t * max_angle)
k = 1.8
r1 = 1.4
r = r1 * np.exp(-k * t)
x1 = 0
y1 = 0
X = x1 + r * np.cos(theta)
Y = y1 + r * np.sin(theta)


normals = pol2cart((1, theta + (np.pi/2)))
X2 = X + normals.real
Y2 = Y + normals.imag


minf = 20
maxf = 20000
minx = np.log10(minf)
maxx = np.log10(maxf)
linx = t * (maxx - minx) + minx
# print(linx)


# fig, ax = plt.subplots(1,1, figsize=(12,6))
thickness = np.geomspace(0.35, 0.06, 100)
norm1 = pol2cart((thickness, theta))
norm2 = pol2cart((-thickness, theta))
X1 = X + norm1.real
Y1 = Y + norm1.imag
X2 = X + norm2.real
Y2 = Y + norm2.imag

X *= -1
X1 *= -1
X2 *= -1

def line_length(x, y):
    deltax = np.diff(x)
    deltay = np.diff(y)
    return np.sum(np.hypot(deltax, deltay))


l = line_length(X, Y)
major_ticks = np.linspace(-l, 0, 4)

def to_points(x, y):
    if hasattr(x, '__iter__'):
        return " ".join([f"({xx:.3f},{yy:.3f})" for xx,yy in zip(x,y)])
    else:
        return f"({x:.3f},{y:.3f})"


tikz = r"""
\begin{tikzpicture}
\begin{axis}[
    xmode=log,  
    ymax=2,
    ymin=-0.5,
    xmin=10, 
    xmax=50000, 
    width=15cm, 
    height=6cm,
    axis x line*=bottom,
    axis y line=none,
    xlabel={Frequency (Hz)},
    xtick={20, 200, 2000, 20000},
    xticklabels={20, 200, 2k, 20k}
]

    \coordinate (base) at (axis cs:20000,0);
    \coordinate (apex) at (axis cs:20,0);
    \coordinate (base_upper) at (axis cs:20000,0.2);
    \coordinate (base_lower) at (axis cs:20000,-0.2);
    \coordinate (apex_upper) at (axis cs:20,0.1);
    \coordinate (apex_lower) at (axis cs:20,-0.1);

    \draw[black] (base_lower) to[bend right=90, looseness=1.618] (base);
    \draw[black] (base) to[bend right=90, looseness=1.618] (base_upper);
    \draw[black] (apex_lower) to[bend left=90, looseness=1.618] (apex_upper);
    \draw[black] (base_upper) -- (apex_upper);
    \draw[black] (base_lower) -- (apex_lower);
    \draw[black, dashed] (base) -- (apex);

    \draw[black,-Latex] (axis cs: 2000, 0.7) node[inner sep=5pt, anchor=south] {Basilar membrane} -- (axis cs: 2000, 0);
\end{axis}

\coordinate (shift) at ($(base) + (0,GOUP)$);
\begin{scope}[shift={(shift)}]
\draw[black, densely dotted] plot[smooth, tension=0.618] coordinates {SPIRAL_UPPER};
\draw[black, densely dotted] plot[smooth, tension=0.618] coordinates {SPIRAL_LOWER};
\draw[black, densely dotted] SPIRAL_APEX1 to[bend left=90, looseness=1.618] SPIRAL_APEX2;
\end{scope}

\node[black] at (base) [inner sep=10pt, anchor=west] {Base};
\node[black] at (apex) [inner sep=10pt, anchor=east] {Apex};

\end{tikzpicture}
"""

tikz = tikz.replace("GOUP", str(r1))
tikz = tikz.replace("SPIRAL_UPPER", to_points(X1, Y1))
# tikz = tikz.replace("SPIRAL_INNER", to_points(X, Y))
tikz = tikz.replace("SPIRAL_LOWER", to_points(X2, Y2))
tikz = tikz.replace("SPIRAL_APEX1", to_points(X1[-1], Y1[-1]))
tikz = tikz.replace("SPIRAL_APEX2", to_points(X2[-1], Y2[-1]))

print(tikz)
Path("../../papers/kappa/figures/unfurled_cochlea.tikz").write_text(tikz)

%tikz -e -t=pgfplots -l="calc,arrows.meta" "$tikz"




In [ ]:
from algorithms.tardieu import make_subbands, extract_envelope, make_cochleagram

audio = pf.signals.exponential_sweep_time(4410, [100, 20000])
# audio = pf.io.read_audio("/Users/richard/Music/freesound/31912__acclivity__globalwarming_marg.wav")
# audio = pf.dsp.time_crop(audio, [0.8, 2.6], unit='s')
# audio.time *= 6
subbands = make_subbands(audio, 8)[0] # shape (channel, subband, time)
envelope = extract_envelope(subbands, audio.sampling_rate) # shape (subband, time)
coch = make_cochleagram(envelope, 1024, 512)


def _plot_subbands(ax, times, signal, separation=1.):
    N = signal.shape[0]
    img = ax.plot(times, np.flip(signal, axis=0).T + np.arange(len(signal) * separation), color='k', linewidth=0.3)
    ax.set_yticks(np.arange(N), labels=np.arange(N))
    return img


# Plotting all audio points on tiny graphs breaks a lot of stuff.
# Downsample audio for display
def _downsample(x: NDArray[np.float64]) -> NDArray[np.float64]:
    # q = 2
    # n = 0
    # for _ in range(n):
    #     x = scipy.signal.decimate(x, q, axis=1)
    # x = x[:,::32]
    times = np.linspace(0,1,x.shape[1])
    return times, x


fig, ax = plt.subplots(1,3, figsize=(12,3))

_plot_subbands(ax[0], *_downsample(subbands))
_plot_subbands(ax[1], *_downsample(envelope))
# _plot_subbands(ax[2], np.arange(coch.shape[1]), coch)
ax[2].imshow(coch, aspect='auto', extent = (0, 1, 0, 8), cmap='magma')
ax[2].set_yticks(np.arange(8) + 0.5, np.arange(8))

ax[0].set_title("Gammatone subbands")
ax[1].set_title("Envelopes")
ax[2].set_title("Cochleagram")

def _make_data(times, signal, downsample=None):
    altered_data = np.flip(signal, axis=0) + np.expand_dims(np.arange(signal.shape[0]), axis=0).T
    all_data = np.stack([times, *altered_data]).T
    if downsample:
        return all_data[::downsample]
    return all_data

np.savetxt("../../papers/kappa/tests/subband_data.dat", _make_data(audio.times, subbands), fmt="%.3e")
np.savetxt("../../papers/kappa/tests/envelope_data.dat", _make_data(audio.times, envelope, downsample=8), fmt="%.3e")
plt.imsave("../../papers/kappa/tests/cochleagram.png", coch)

